# Create Base Dataset

Builds the denormalised base dataset used as the starting point for all
downstream data science steps.

**Input:** Star-schema tables from `data/gold/data_warehouse/`
**Output:** `data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet`

**Steps:**
1. Load dimension and fact tables from the data warehouse.
2. Generate a full weekly spine for every time series (no gaps).
3. Enrich with region, product, supplier, and calendar attributes.
4. Optimise column ordering and data types to reduce memory footprint.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
# -- 1. Load data warehouse tables ----------------------------------------
DW = '../../data/gold/data_warehouse'

df_dim_region      = pd.read_parquet(f'{DW}/dw_dim_region.parquet')
df_dim_product     = pd.read_parquet(f'{DW}/dw_dim_product.parquet')
df_dim_supplier    = pd.read_parquet(f'{DW}/dw_dim_supplier.parquet')
df_dim_calendar    = pd.read_parquet(f'{DW}/dw_dim_weekly_calendar.parquet')
df_dim_time_series = pd.read_parquet(f'{DW}/dw_dim_time_series.parquet')
df_fact_sales      = pd.read_parquet(f'{DW}/dw_fact_sales_weekly.parquet')

print(f'Time series registry : {len(df_dim_time_series):,} series')
print(f'Fact sales           : {len(df_fact_sales):,} rows')

In [ ]:
# -- 2. Generate full weekly spine (one row per series x week) ---------------
def generate_spines_time_series(
    df_fact_sales: 'pd.DataFrame',
    df_dim_time_series: 'pd.DataFrame',
) -> 'pd.DataFrame':
    rows = []
    for _, r in df_dim_time_series.iterrows():
        start = pd.to_datetime(r['first_week_date'])
        end   = pd.to_datetime(r['last_week_date'])
        if pd.isna(start) or pd.isna(end):
            continue
        for d in pd.date_range(start=start, end=end, freq='W-MON'):
            rows.append({'time_series_id': r['time_series_id'], 'week_date': d})

    df_spine = pd.DataFrame(rows)
    df_meta  = df_dim_time_series.drop(columns=['first_week_date', 'last_week_date'], errors='ignore')
    df_spine = df_spine.merge(df_meta, on='time_series_id', how='left')

    df_out = df_spine.merge(
        df_fact_sales[['time_series_id', 'week_date', 'units_sold']],
        on=['time_series_id', 'week_date'],
        how='left',
    )
    df_out['units_sold'] = df_out['units_sold'].fillna(0).astype(int)
    return df_out


df = generate_spines_time_series(df_fact_sales, df_dim_time_series)
print(f'Spine: {len(df):,} rows  |  {df["time_series_id"].nunique():,} series')

In [ ]:
# -- 3. Enrich with dimension attributes ------------------------------------
def bring_dimensions(df, df_dim_region, df_dim_product, df_dim_supplier, df_dim_calendar):
    return (
        df
        .merge(df_dim_region,   on='region_id',   how='left')
        .merge(df_dim_product,  on='product_id',  how='left')
        .merge(df_dim_supplier, on='supplier_id', how='left')
        .merge(df_dim_calendar, on='week_date',   how='left')
    )


df = bring_dimensions(df, df_dim_region, df_dim_product, df_dim_supplier, df_dim_calendar)
print(f'Shape after enrichment: {df.shape}')

In [ ]:
# -- 4. Optimise column order and data types --------------------------------
def optimize_base_dataset(df: 'pd.DataFrame') -> 'pd.DataFrame':
    df = df.copy()
    column_order = [
        'time_series_id', 'week_date',
        'supplier_id', 'region_id', 'product_id',
        'units_sold',
        'week', 'start_date', 'end_date', 'year', 'semester', 'semester_date',
        'semester_name', 'quarter', 'quarter_date', 'quarter_name',
        'month', 'month_name', 'month_date',
        'first_week_date', 'last_week_date', 'total_weeks_length',
        'num_week_with_sales', 'num_week_with_zeros', 'sales_weeks_ratio',
        'sales_units', 'avg_weekly_sales', 'avg_weekly_sales_non_zero',
        'std_weekly_sales', 'std_weekly_sales_non_zero',
        'max_weekly_sales', 'min_weekly_sales',
        'q25_sales', 'q50_sales', 'q75_sales', 'iqr', 'cv',
        'supplier_name', 'region_name', 'product_name',
        'product_attribute_1', 'product_attribute_2', 'product_attribute_3',
    ]
    existing_order = [c for c in column_order if c in df.columns]
    remaining      = [c for c in df.columns if c not in existing_order]
    df = df[existing_order + remaining]

    if 'time_series_id' in df.columns:
        df['time_series_id'] = df['time_series_id'].astype('int32')
    for col in ['supplier_id', 'region_id', 'product_id']:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in ['week_date', 'first_week_date', 'last_week_date',
                'start_date', 'end_date', 'semester_date', 'quarter_date', 'month_date']:
        if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col])
    for col in ['year', 'semester', 'quarter', 'month', 'week']:
        if col in df.columns:
            df[col] = df[col].astype('int16')
    for col in ['semester_name', 'quarter_name', 'month_name']:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in ['units_sold', 'sales_units', 'max_weekly_sales', 'min_weekly_sales',
                'q25_sales', 'q50_sales', 'q75_sales',
                'num_week_with_sales', 'num_week_with_zeros', 'total_weeks_length']:
        if col in df.columns:
            df[col] = df[col].astype('Int64')
    for col in ['avg_weekly_sales', 'std_weekly_sales', 'cv', 'iqr', 'sales_weeks_ratio']:
        if col in df.columns:
            df[col] = df[col].astype('float32')
    for col in ['supplier_name', 'region_name', 'product_name',
                'product_attribute_1', 'product_attribute_2', 'product_attribute_3']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'Shape  : {df.shape}')
    print(f'Memory : {mem_mb:.1f} MB')
    return df


df_base = optimize_base_dataset(df)

In [ ]:
# -- 5. Save ----------------------------------------------------------------
import os

OUT_PATH = '../../data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df_base.to_parquet(OUT_PATH, index=False)

print(f'Saved  : {OUT_PATH}')
print(f'Rows   : {len(df_base):,}')
print(f'Series : {df_base["time_series_id"].nunique():,}')